# Read and Inspect a CSV File

`CSV.File` parses a delimited text file into typed columns. This tutorial writes a few small sample files, reads them with the right options and checks that every column came out as expected.

It needs CSV.jl, Tables.jl and Plots.jl. All files are created in a temporary folder.

In [ ]:
Pollis.packages("CSV", "Tables", "Plots")

In [ ]:
using CSV, Tables, Plots, Dates

cd(mktempdir())
pwd()

## A Sample File

A typical export: a header row, numbers with some missing values written as `NA`, text, dates and booleans.

In [ ]:
write("sales.csv", """
id,price,city,date,paid
1,9.50,Lisbon,2024-01-05,true
2,NA,Porto,2024-01-06,false
3,12.00,Faro,2024-01-07,true
4,7.25,Lisbon,2024-01-08,true
5,,Porto,2024-01-09,false
""")
print(read("sales.csv", String))

## Read It

Without options, CSV.jl detects the delimiter, takes the column names from the first line and treats empty fields as missing. `NA` is not a missing marker by default, so the `price` column has to be read as text.

In [ ]:
file = CSV.File("sales.csv")
Tables.schema(file)

List every missing marker in `missingstring`. Listing markers replaces the default, so include `""` to keep empty fields missing.

In [ ]:
file = CSV.File("sales.csv"; missingstring = ["", "NA"])
Tables.schema(file)

## Work with the Columns

A `CSV.File` is a Tables.jl table. `Tables.columntable` turns it into a named tuple of vectors, the simplest table in Julia.

In [ ]:
tbl = Tables.columntable(file)
println("Rows: ", length(first(tbl)), ", columns: ", keys(tbl))
println("Prices: ", tbl.price)
println("Mean price paid: ", sum(skipmissing(tbl.price[tbl.paid])) / count(!ismissing, tbl.price[tbl.paid]))

In [ ]:
bar(string.(tbl.date), coalesce.(tbl.price, 0.0); label = "price", xlabel = "date", xrotation = 30, title = "Sales")

## A European Export

Semicolons between fields, a decimal comma and day-first dates. Read with defaults, everything is wrong: the numbers and dates stay text.

In [ ]:
write("eu.csv", """
id;price;date
1;9,50;05/01/2024
2;12,00;06/01/2024
3;7,25;07/01/2024
""")
Tables.schema(CSV.File("eu.csv"))

In [ ]:
eu = CSV.File("eu.csv"; delim = ';', decimal = ',', dateformat = "dd/mm/yyyy")
Tables.schema(eu)

In [ ]:
Tables.columntable(eu)

## Read Only What You Need

On wide or long files, `select` keeps some columns and `limit` stops after a number of rows. `types` forces a column type, for example to keep the leading zeros of codes that look like numbers.

In [ ]:
write("codes.csv", """
code,name,value
00123,alpha,1
04567,beta,2
08910,gamma,3
""")
println(CSV.File("codes.csv").code)                                  # inferred as integers: zeros lost
println(CSV.File("codes.csv"; types = Dict(:code => String)).code)   # kept as text
println(CSV.File("codes.csv"; select = [:code, :value], limit = 2, types = Dict(:code => String)))

## Check the Result

When a single value in a numeric column cannot be parsed, the whole column becomes text. `strict = true` stops at the first such value and says where it is.

In [ ]:
write("bad.csv", """
x,y
1,2
n/a,3
""")
println(Tables.schema(CSV.File("bad.csv")))
try
	CSV.File("bad.csv"; types = Dict(:x => Int), strict = true)
catch err
	println(sprint(showerror, err))
end

## Next Steps

- Tutorial 2 writes tables back to disk and checks the round trip.
- Tutorial 3 handles files too large to read at once.
- The panel's **Schema Check** and **Strict Parse** actions run these checks on your own file.